# How to Create Loops with Claude Code: A Practical Guide to Agentic Automation

Run from the companion repository root. See the adjacent topic folder README for pinned setup and execution boundaries. Outputs are intentionally not committed.

Cloudflare live requests, Claude sessions/scheduling and Hermes messaging/cron are not part of the default checks.

In [ ]:
from pathlib import Path
import tempfile,sys,json,subprocess,hashlib
folder=Path.cwd()/'claude-code-loops'
if not folder.exists():folder=Path.cwd()
sys.path.insert(0,str(folder))
from verify_loop import snapshot,verify,REPORT_SECTIONS,STATE_SECTIONS
workspace=Path(tempfile.mkdtemp(prefix='tdb-claude-loop-'))
print('Owned fictional workspace created; no Claude model or scheduler invoked.')

In [ ]:
for name in ['TASK.md','PROGRESS.md','LOOP_INSTRUCTIONS.md']:
    (workspace/name).write_text('# '+name+'\nManual fixture setup.\n')
(workspace/'outputs').mkdir()
(workspace/'outputs/daily-review.md').write_text('# Daily Review\nNo run yet.\n')
(workspace/'notes.md').write_text('Fictional project: documentation review only.\n')
before=snapshot(workspace)
assert verify(workspace,before)['accepted'] is False
print('Starter state correctly fails the completion gate.')

In [ ]:
# Deterministic fixture worker: test the checker, not Claude's reasoning.
report='# Daily Review\n'+''.join('\n## '+section+'\nFixture observation: '+section+'.\n' for section in REPORT_SECTIONS)
progress='# Loop Progress\n'+''.join('\n## '+section+'\nManual fixture run; review before scheduling.\n' for section in STATE_SECTIONS)
(workspace/'outputs/daily-review.md').write_text(report)
(workspace/'PROGRESS.md').write_text(progress)
result=verify(workspace,before)
assert result['accepted'] and result['changed']==['PROGRESS.md','outputs/daily-review.md']
print(json.dumps(result,indent=2))

In [ ]:
(workspace/'outputs/daily-review.md').write_text(report.replace('## Blockers or unresolved questions','## Missing'))
assert not verify(workspace,before)['accepted']
(workspace/'outputs/daily-review.md').write_text(report)
(workspace/'notes.md').write_text('Forbidden fixture modification.\n')
assert 'Forbidden file modified: notes.md' in verify(workspace,before)['failed']
(workspace/'notes.md').write_text('Fictional project: documentation review only.\n')
assert verify(workspace,before)['accepted']
stable=snapshot(workspace)
assert 'PROGRESS.md was not updated' in verify(workspace,stable)['failed']
print('Missing section, forbidden edit and unchanged-state rejection passed.')

In [ ]:
# Execute the walkthrough's actual mkdir/touch commands inside a separate owned directory.
shell_root=Path(tempfile.mkdtemp(prefix='tdb-loop-shell-'))
commands='mkdir minimal-claude-loop\ncd minimal-claude-loop\ntouch TASK.md PROGRESS.md LOOP_INSTRUCTIONS.md\nmkdir outputs\ntouch outputs/daily-review.md'
subprocess.run(['bash','-eu','-c',commands],cwd=shell_root,check=True)
assert sorted(p.name for p in (shell_root/'minimal-claude-loop').iterdir())==['LOOP_INSTRUCTIONS.md','PROGRESS.md','TASK.md','outputs']
print('Actual Bash workspace setup passed in an owned temporary directory.')

In [ ]:
outside=Path(tempfile.mkdtemp(prefix='tdb-loop-outside-'))/'untouched.txt'
outside.write_text('Outside fixture, not read by checker.\n')
digest=hashlib.sha256(outside.read_bytes()).hexdigest()
(workspace/'unexpected-link').symlink_to(outside)
try:snapshot(workspace)
except ValueError:pass
else:raise AssertionError('Symlink accepted')
assert hashlib.sha256(outside.read_bytes()).hexdigest()==digest
print('Symlink rejected without reading or changing its target. No schedules or external messages created.')

## Original archive appendix

Complete original blocks and transcripts, preserved for comparison. These Markdown fences are not executable cells.

### Original block 1

````text
project-review-loop/
├── TASK.md
├── PROGRESS.md
├── LOOP_INSTRUCTIONS.md
└── outputs/
    └── daily-review.md
````

### Original block 2

````text
Example #1: Daily project review:
Read the project folder every morning, summarize what changed, identify blockers, update PROGRESS.md, and write a report to outputs/daily-review.md.

Example #2: Meeting follow-up loop:
After a meeting notes file is added, extract action items, owners, deadlines, and unresolved questions into outputs/follow-up.md.

Example #3: Folder cleanup planning loop:
Inspect a messy folder, classify files by type, propose a cleanup plan, and write the plan to outputs/cleanup-plan.md. Do not move or rename files yet.

Example #4:CI failure triage loop:
When CI fails, read the logs, classify the failure, identify likely causes, and draft a report. Do not change code until a human approves the suggested fix.
````

### Original block 3

````text
Example #1: Rewrite the entire backend architecture.

Example #2: Improve our product direction every week.

Example #3: Automatically modify authentication and payment code until all issues are fixed.

Example #4: Deploy changes to production when Claude thinks they are ready.

Example #5: Clean my whole computer and reorganize all files automatically.
````

### Original block 4

````text
Level 1 — Read-only analysis
Claude reads files and writes a summary.

Level 2 — Draft output
Claude writes reports, plans, or suggested changes in an outputs/ folder.

Level 3 — Sandbox edits
Claude modifies files only inside a controlled sandbox or draft branch.

Level 4 — Draft external action
Claude opens a draft pull request, draft message, or draft ticket update.

Level 5 — Human-approved action
Claude applies changes only after explicit approval.

Level 6 — Fully automated low-risk action
Claude completes the task automatically, with logs, limits, and rollback options.
````

### Original block 5

````text
Every morning, review this project folder, compare it with the previous state, summarize changes, identify blockers, update PROGRESS.md, and write a daily report to outputs/daily-review.md.
````

### Original block 6

````text
minimal-claude-loop/
├── TASK.md
├── PROGRESS.md
├── LOOP_INSTRUCTIONS.md
└── outputs/
    └── daily-review.md
````

### Original block 7

````text
# Daily Project Review Loop

## Goal
Review this project folder, summarize what changed, identify blockers, and produce a short daily review report.

## Expected Output
Each run should produce or update:
- `outputs/daily-review.md`
- `PROGRESS.md`


## Scope
Claude may inspect files in this workspace and write reports to the `outputs/` folder.
Claude should not modify source files, delete files, rename files, or move files during this first version of the loop.
````

### Original block 8

````text
# Loop Progress

## Last Run
- Date:
- Summary:
- Files checked:
- Output produced:

## Current Status
- In progress:
- Blocked:
- Needs human review:

## Next Run Should
-
## Decisions Made
-
## Do Not Repeat
-
````

### Original block 9

````text
# Loop Instructions

You are running a daily project review loop.

## Before You Start
1. Read `TASK.md`.
2. Read `PROGRESS.md`.
3. Inspect the project folder.
4. Identify what changed, what is incomplete, and what needs human review.

## What You Should Do
Write a short daily review report to:
`outputs/daily-review.md`
The report should include:
- Summary of current project state
- Files or areas reviewed
- Meaningful changes found
- Blockers or unresolved questions
- Recommended next actions

After writing the report, update `PROGRESS.md` with:
- Date of this run
- Summary of what happened
- Files checked
- Output produced
- What the next run should do
- Anything that needs human review

## Safety Rules
- Do not delete files.
- Do not rename files.
- Do not move files.
- Do not modify source files.
- Only write to `outputs/daily-review.md` and `PROGRESS.md`.
- If you are unsure whether an action is allowed, stop and ask for human review.

## Verification Checklist
Before ending the run, check that:
- `outputs/daily-review.md` exists.
- `outputs/daily-review.md` contains all required sections.
- `PROGRESS.md` has been updated.
- No files outside `outputs/daily-review.md` and `PROGRESS.md` were modified.
````

### Original block 10

````text
# Daily Review

No loop run has been completed yet.
````

### Original block 11

````text
outputs/
├── daily-review-2026-06-22.md
├── daily-review-2026-06-23.md
└── daily-review-2026-06-24.md
````

### Original block 12

````text
Run the daily project review loop for this workspace.

Follow `LOOP_INSTRUCTIONS.md` exactly.
Before acting, read `TASK.md` and `PROGRESS.md`.
Write the daily review to `outputs/daily-review.md`, update `PROGRESS.md`, and then report what you changed.
Do not modify any files except `outputs/daily-review.md` and `PROGRESS.md`.
````

### Original block 13

````text
- Did Claude read TASK.md?
- Did Claude read PROGRESS.md?
- Did Claude write outputs/daily-review.md?
- Did Claude update PROGRESS.md?
- Did Claude avoid modifying files outside the allowed paths?
- Is the daily review structured and useful?
- Is the next action clear?
````

### Original block 14

````text
- Did the test suite pass?
- Did the type checker pass?
- Did the linter pass?
- Did the build complete successfully?
- Did Claude only change files inside the allowed scope?
````

### Original block 15

````text
- Does the output include all required sections?
- Are unresolved questions listed separately?
- Are action items assigned clearly?
- Is the output saved in the expected location?
- Did Claude avoid editing the original document?
````

### Original block 16

````text
Trigger:
A manual prompt starts the run.

Context:
Claude reads TASK.md, PROGRESS.md, and LOOP_INSTRUCTIONS.md.

Action:
Claude inspects the workspace and writes a daily report.

Verification:
Claude checks the required files and allowed modification scope.

State update:
Claude updates PROGRESS.md.

Decision:
The loop stops and waits for the next manual or scheduled run.
````

### Original block 17

````text
What happened during the last run?
Which files were reviewed?
What output was produced?
What is currently blocked?
What should the next run focus on?
What should Claude avoid repeating?
What still needs human review?
````

### Original block 18

````text
# Loop Progress

## Current State
- Status:
- Main objective:
- Current focus:
- Last updated:

## Last Run
- Date:
- Trigger:
- Summary:
- Files reviewed:
- Output produced:

## Open Items
- 

## Blockers
- 

## Needs Human Review
- 

## Next Run Should
- 

## Decisions Made
- 

## Do Not Repeat
-
````

### Original block 19

````text
- The latest known status of the loop
- The files or areas checked in the last run
- Open tasks that still need work
- Blockers or missing information
- Decisions made by the human
- Instructions not to repeat failed actions
- The recommended focus for the next run
````

### Original block 20

````text
- Full transcripts of Claude conversations
- Long generated reports
- Repeated copies of the same status update
- Large code snippets
- Raw logs
- Unfiltered brainstorming notes
````

### Original block 21

````text
# Loop Progress

## Current State
- Status: Active manual testing
- Main objective: Produce a daily project review report
- Current focus: Verify that Claude reads the instruction files and writes only to allowed paths
- Last updated: 2026-06-22
## Last Run
- Date: 2026-06-22
- Trigger: Manual run
- Summary: Claude reviewed the workspace structure and generated the first daily review report.
- Files reviewed:
  - TASK.md
  - LOOP_INSTRUCTIONS.md
  - PROGRESS.md
  - outputs/daily-review.md
- Output produced:
  - outputs/daily-review.md

## Open Items
- Run the loop manually two more times before scheduling.
- Add a clearer output format if the report becomes too verbose.

## Blockers
- None.
## Needs Human Review
- Confirm whether the daily review format is useful.
- Confirm whether future runs should create timestamped reports or overwrite `daily-review.md`.
## Next Run Should
- Read this file first.
- Check whether any project files were added or changed.
- Update the daily review report.
- Keep the output concise.
## Decisions Made
- The loop is allowed to write only to `outputs/daily-review.md` and `PROGRESS.md`.
- Scheduling should not be enabled until the manual run is stable.
## Do Not Repeat
- Do not modify source files.
- Do not rename or move files.
````

### Original block 22

````text
outputs/
├── daily-review.md
├── history/
│   ├── 2026-06-22-review.md
│   ├── 2026-06-23-review.md
│   └── 2026-06-24-review.md
````

### Original block 23

````text
## State Update Rule

Before ending every run, update `PROGRESS.md`.
The update must include:
- What happened in this run
- Which files were reviewed
- Which output files were created or changed
- Any blockers or unresolved questions
- What the next run should do
- Whether human review is needed
If you cannot update `PROGRESS.md`, stop and report the reason.
````

### Original block 24

````text
TASK.md:
What is the loop trying to accomplish?

LOOP_INSTRUCTIONS.md:
How should Claude run the loop?

PROGRESS.md:
What is the current state of the loop?

outputs/:
What did the loop produce?
````

### Original block 25

````text
Markdown state:
Best for solo workflows, local projects, writing workflows, and simple code loops.

GitHub Issues or Linear:
Best when loop state needs to be visible to a team.

Database or JSON state:
Best when the loop needs structured querying, multiple concurrent runs, or programmatic state updates.

Slack or email summaries:
Best for human-facing notifications, but not ideal as the primary state source.
````

### Original block 26

````text
## Do Not Repeat

- Do not rerun the same cleanup plan until the human approves it.
- Do not modify files outside the outputs/ folder.
- Do not suggest deleting files unless they are duplicates confirmed by name and checksum.
````

### Original block 27

````text
## Needs Human Review

- Confirm whether archived files should be moved or left in place.
- Confirm whether the loop should switch from overwriting daily-review.md to creating timestamped reports.
````

### Original block 28

````text
- The report looks good.
- The task seems complete.
- Claude says the issue is fixed.
- The output is probably ready.
````

### Original block 29

````text
- outputs/daily-review.md exists.
- outputs/daily-review.md contains all required sections.
- PROGRESS.md has been updated.
- No files outside the allowed paths were modified.
- The output includes blockers and next actions.
- If uncertainty exists, the run is marked as needing human review.
````

### Original block 30

````text
- Unit tests pass.
- Type checks pass.
- Linting passes.
- The build succeeds.
- No forbidden files were changed.
- A draft pull request was created instead of merging directly.
````

### Original block 31

````text
Worker phase:
- Read TASK.md, PROGRESS.md, and LOOP_INSTRUCTIONS.md.
- Inspect the workspace.
- Write outputs/daily-review.md.
- Update PROGRESS.md.

Checker phase:
- Confirm outputs/daily-review.md exists.
- Confirm all required report sections are present.
- Confirm PROGRESS.md was updated.
- Confirm no forbidden files were modified.
- If any check fails, fix the issue or mark the run as needing human review.
````

### Original block 32

````text
## Verification Checklist

Before ending the run, verify the following:
### Required Files
- `outputs/daily-review.md` exists.
- `PROGRESS.md` exists.

### Required Report Sections
`outputs/daily-review.md` must include:
- Summary
- Files reviewed
- Meaningful changes
- Blockers or unresolved questions
- Recommended next actions

### State Update
`PROGRESS.md` must include:
- Date of the current run
- Summary of what happened
- Files checked
- Output produced
- What the next run should do
- Whether human review is needed

### Safety Boundary
Only these files may be modified:
- `outputs/daily-review.md`
- `PROGRESS.md`
If any other file was modified, stop and report the issue.
````

### Original block 33

````text
Run a verification pass for the latest loop execution.

Check the result against the Verification Checklist in LOOP_INSTRUCTIONS.md.
Report:
1. Which checks passed
2. Which checks failed
3. Which files were modified
4. Whether the run is safe to accept
5. Whether human review is required
Do not modify any files during this verification pass.
````

### Original block 34

````text
- Action items are listed separately.
- Each action item has an owner if one was mentioned.
- Deadlines are extracted when present.
- Open questions are separated from confirmed decisions.
- The original meeting notes were not modified.
````

### Original block 35

````text
- No files were moved or renamed.
- Files were grouped by type.
- Potential duplicates were marked for review.
- The cleanup plan was written to outputs/cleanup-plan.md.
- Risky actions were listed under “Needs Human Review.”
````

### Original block 36

````text
- Each claim has a source link or note.
- Open uncertainties are listed.
- Draft ideas are separated from verified facts.
- The loop does not publish or send anything automatically.
````

### Original block 37

````text
Verification commands:

- npm test
- npm run lint
- npm run typecheck
- npm run build
````

### Original block 38

````text
Verification commands:

- pytest
- ruff check .
- mypy .
````

### Original block 39

````text
## Code Verification

If this loop modifies code, run the following checks before marking the run complete:
```bash
npm test
npm run lint
npm run typecheck
````

### Original block 40

````text
The key point is that the loop should not rely only on generated reasoning. It should use the project’s existing verification infrastructure. Tests, type checkers, linters, and builds are valuable because they can fail the work without needing Claude to decide subjectively.

### Preventing the Checker from Becoming Another Optimist
A weak checker simply reads the worker's output and agrees with it. This is common when the checker prompt is vague:
```text id="epfh9h"
Review the output and tell me if it looks good.
````

### Original block 41

````text
Evaluate the latest loop output against the checklist below.

For each item, return PASS or FAIL.
Do not give partial credit.
Do not assume a missing section is present.
Do not mark the run complete if any required item fails.
Checklist:
- outputs/daily-review.md exists
- Summary section exists
- Files reviewed section exists
- Blockers section exists
- Recommended next actions section exists
- PROGRESS.md was updated
- No files outside the allowed paths were modified
If any item fails, mark the run as NOT ACCEPTED.
````

### Original block 42

````text
1. Retry
Use when the failure is small, clear, and safe to fix.

2. Escalate
Use when the failure needs human judgment or touches a risky area.

3. Stop
Use when the loop hit its iteration limit, permission boundary, or budget limit.
````

### Original block 43

````text
## Failure Policy

If verification fails:
1. If the failure is a missing section in `outputs/daily-review.md`, fix the report once.
2. If `PROGRESS.md` was not updated, update it once.
3. If any forbidden file was modified, stop immediately and report the issue.
4. If the same verification check fails twice, stop and mark the run as needing human review.
````

### Original block 44

````text
Maximum attempts: 2
If verification still fails after 2 attempts, stop and ask for human review.
````

### Original block 45

````text
Maximum attempts per scheduled run: 3
Maximum runtime: 20 minutes
Maximum files modified: 3
Stop immediately if a forbidden path is touched.
````

### Original block 46

````text
## Last Verification

- Date: 2026-06-23
- Result: Failed
- Failed checks:
  - Missing "Blockers" section in outputs/daily-review.md
- Action taken:
  - Report was corrected once
- Final status:
  - Passed after correction
````

### Original block 47

````text
## Last Verification

- Date: 2026-06-23
- Result: Failed
- Failed checks:
  - Forbidden file modified: src/config.ts
- Action taken:
  - Stopped immediately
- Final status:
  - Needs human review
````

### Original block 48

````text
- A verification checklist in LOOP_INSTRUCTIONS.md
- A manual verification prompt
- A rule that failed verification must be recorded in PROGRESS.md
- A rule that repeated failure escalates to human review
````

### Original block 49

````text
/loop 24h Run the daily project review loop for this workspace.

Follow LOOP_INSTRUCTIONS.md exactly.
Before acting:
- Read TASK.md
- Read PROGRESS.md
- Inspect the workspace

Then:
- Write the daily review to outputs/daily-review.md
- Update PROGRESS.md
- Run the verification checklist
- Stop if human review is required
Do not modify any files except outputs/daily-review.md and PROGRESS.md.
````

### Original block 50

````text
/loop 15m Run the daily project review loop for this workspace. Follow LOOP_INSTRUCTIONS.md exactly.
````

### Original block 51

````text
/loop 24h Run the daily project review loop for this workspace. Follow LOOP_INSTRUCTIONS.md exactly.
````

### Original block 52

````text
/loop 7d Inspect the workspace for files that may need cleanup. Write a cleanup plan to outputs/cleanup-plan.md and update PROGRESS.md. Do not move, rename, or delete files.
````

### Original block 53

````text
/loop 30m Check whether the current pull request has new CI failures or review comments. Summarize findings in PROGRESS.md and do not modify code unless explicitly instructed.
````

### Original block 54

````text
/loop 10m Check whether the deployment has completed. If it succeeds, summarize the result and stop. If it fails, record the error and mark the run as needing human review.
````

### Original block 55

````text
/loop 24h Run the daily project review loop.
````

### Original block 56

````text
/goal outputs/daily-review.md exists, PROGRESS.md is updated, the verification checklist passes, and no files outside the allowed paths were modified.
````

### Original block 57

````text
- Check every morning
- Poll every 30 minutes
- Review the folder once per week
- Check the PR until new information appears
````

### Original block 58

````text
- Continue until tests pass
- Continue until the report has all required sections
- Continue until the issue queue is empty
- Continue until the output matches the acceptance criteria
````

### Original block 59

````text
## Scheduled Run Policy

When this loop runs on a schedule:
- If there are meaningful changes, write a concise report to `outputs/daily-review.md`.
- If there are no meaningful changes, write a short "No meaningful changes" note.
- If human review is needed, mark it clearly in `PROGRESS.md`.
- If the same blocker appears in two consecutive runs, do not keep retrying. Escalate to human review.
- If verification fails twice, stop and mark the run as not accepted.
````

### Original block 60

````text
Quiet mode:
Nothing meaningful changed. Update state briefly and stop.

Attention mode:
Something changed, failed, or needs review. Write a clear report and flag the issue.
````

### Original block 61

````text
Day 1:
Run manually and verify output.

Day 2:
Run with a short /loop testing interval while watching.

Day 3:
Move to the real cadence.

Days 4–7:
Review every scheduled output.

After one week:
Decide whether to keep, adjust, or expand the loop.
````

### Original block 62

````text
1. Useful outputs
How many runs produced something worth reading?

2. False alarms
How many runs created noise or flagged issues that did not matter?

3. Missed issues
How many important changes did the loop fail to notice?
````

### Original block 63

````text
## Loop Review Notes

- 2026-06-24: Output useful. Summary was concise.
- 2026-06-25: Too verbose. Reduced report length in LOOP_INSTRUCTIONS.md.
- 2026-06-26: Missed new file in docs/. Added instruction to inspect docs/ folder.
````

### Original block 64

````text
Allowed:
- Read workspace files
- Write outputs/daily-review.md
- Update PROGRESS.md

Not allowed:
- Delete files
- Rename files
- Move files
- Modify source files
- Send messages
- Open external tickets
- Push code
````

### Original block 65

````text
- The loop must run even when the current Claude Code session is closed.
- The loop must run when your machine is off.
- The loop should be triggered by an external event rather than time.
- The loop needs production-grade observability and logs.
- The loop is part of a CI/CD pipeline.
````

### Original block 66

````text
/loop 24h Run the daily project review loop for this workspace.

Follow LOOP_INSTRUCTIONS.md exactly.
Read TASK.md and PROGRESS.md first.
Write the report to outputs/daily-review.md.
Update PROGRESS.md before stopping.
Run the verification checklist.
If there are no meaningful changes, keep the report short.
If human review is needed, clearly mark it in PROGRESS.md.
Do not modify any files except outputs/daily-review.md and PROGRESS.md.
````

### Original block 67

````text
Allowed:
- Read files in the workspace
- Write outputs/daily-review.md
- Update PROGRESS.md

Not allowed:
- Delete files
- Rename files
- Move files
- Modify source files
- Send messages
- Open tickets
- Push code
````

### Original block 68

````text
GitHub:
Read issues, inspect pull requests, check CI status, open draft PRs.

Linear or Jira:
Read tickets, update status, link work to tasks, create follow-up items.

Slack:
Post summaries, notify humans on escalation, share daily reports.

Database or analytics tools:
Read metrics, query logs, inspect operational state.

Sentry or error tracking:
Read recent errors, group recurring failures, draft investigation notes.
````

### Original block 69

````text
Every morning:
- Read PROGRESS.md
- Inspect the local project folder
- Check open GitHub issues
- Check recent pull requests
- Check failed CI runs
- Write a daily review report
- Update PROGRESS.md
- Flag anything that needs human review
````

### Original block 70

````text
Level 1 — Read-only access
Claude can inspect files, tickets, logs, issues, and dashboards.

Level 2 - Draft outputs
Claude can write reports, draft ticket comments, draft Slack messages, or draft PR descriptions.

Level 3 - Sandbox edits
Claude can modify files only in a sandbox folder, draft branch, or isolated worktree.

Level 4 - Draft external actions
Claude can prepare external actions, such as draft pull requests or draft issue updates, but not finalize them.

Level 5 - Human-approved writes
Claude can perform writes only after explicit human approval.

Level 6 - Fully automated low-risk writes
Claude can complete narrowly scoped actions automatically, with logs, limits, and rollback.
````

### Original block 71

````text
Read open Jira tickets.
Summarize likely blockers.
Draft suggested ticket updates in outputs/jira-updates.md.
Do not update Jira directly.
````

### Original block 72

````text
Draft a Slack update in outputs/slack-draft.md.
Do not send the message.
Mark the draft as needing human review in PROGRESS.md.
````

### Original block 73

````text
Inspect the failing test.
Write an investigation report.
Suggest a fix.
Do not modify code unless explicitly approved.
````

### Original block 74

````text
## Tool Permissions Policy

### Filesystem
Allowed:
- Read files in this workspace
- Write `outputs/daily-review.md`
- Update `PROGRESS.md`
Not allowed:
- Delete files
- Rename files
- Move files
- Modify source files

### GitHub
Allowed:
- Read open issues
- Read pull request status
- Read CI status
- Draft a summary in `outputs/github-review.md`
Not allowed:
- Push commits
- Merge pull requests
- Close issues
- Change labels
- Comment publicly without human approval

### Slack
Allowed:
- Draft a message in `outputs/slack-update.md`
Not allowed:
- Send messages
- Mention users
- Post to channels without human approval

### Linear / Jira
Allowed:
- Read assigned tickets
- Draft suggested status updates
Not allowed:
- Change ticket status
- Assign tickets
- Close tickets
- Create new tickets without approval
````

### Original block 75

````text
Allow:
- Read files in the project workspace
- Write specific output files
- Run specific safe commands if needed

Ask:
- Editing project files
- Opening pull requests
- Posting messages
- Updating external systems

Deny:
- Reading secrets
- Deleting files
- Moving large folders
- Pushing to protected branches
- Running unknown scripts
- Accessing unrelated directories
````

### Original block 76

````text
Allowed commands:
- npm test
- npm run lint
- npm run typecheck
- npm run build
- git status
- git diff
````

### Original block 77

````text
Allowed commands:
- pytest
- ruff check .
- mypy .
- git status
- git diff
````

### Original block 78

````text
- Run any bash command
- Install any package
- Modify any file
- Execute any script found in the repo
````

### Original block 79

````text
## Command Policy

Allowed commands:
- `git status`
- `git diff`
- `npm test`
- `npm run lint`
- `npm run typecheck`

Not allowed:
- Installing packages
- Running arbitrary scripts
- Deleting files
- Changing environment variables
- Reading `.env`, credentials, SSH keys, or cloud config files
If a command outside the allowlist appears necessary, stop and ask for human review.
````

### Original block 80

````text
1. Create or use an isolated branch/worktree.
2. Apply the change only inside that isolated workspace.
3. Run tests and verification commands.
4. Write a summary of the changes.
5. Open a draft PR or ask for human review.
6. Do not merge automatically.
````

### Original block 81

````text
## Sensitive Data Policy

The loop must not read, copy, summarize, or expose:
- `.env` files
- API keys
- SSH keys
- Cloud credentials
- Database passwords
- Customer personal data unless explicitly required and approved
- Private tokens in logs
If sensitive data appears in a file or tool response, do not include it in outputs. Replace it with `[REDACTED]` and mark the run as needing human review.
````

### Original block 82

````text
Slack:
Write outputs/slack-draft.md instead of posting directly.

GitHub:
Open a draft PR instead of merging.

Linear/Jira:
Write suggested ticket updates instead of changing status.

Database:
Write a query result summary instead of modifying records.

Email:
Draft the message instead of sending it.
````

### Original block 83

````text
outputs/
├── daily-review.md
├── audit-log.md
└── history/
    ├── 2026-06-24-review.md
    └── 2026-06-25-review.md
````

### Original block 84

````text
## 2026-06-24 Scheduled Run

- Trigger: `/loop 24h`
- Tools used:
  - Filesystem read
  - GitHub issue read
- Files changed:
  - outputs/daily-review.md
  - PROGRESS.md
- External actions:
  - None
- Human review required:
  - Yes
- Reason:
  - CI failure needs investigation before any code changes
````

### Original block 85

````text
## Permission Review Rule

Review this loop's permissions every 30 days or whenever a new tool is added.
During review, check:
- Which tools the loop can access
- Which files the loop can modify
- Which external systems the loop can write to
- Whether any permission is broader than needed
- Whether human approval is still required before external actions
````

### Original block 86

````text
Run the daily project review loop.

Read:
- TASK.md
- LOOP_INSTRUCTIONS.md
- PROGRESS.md
Inspect:
- Local workspace
- Open GitHub issues
- Recent pull requests
- CI status
Write:
- outputs/daily-review.md
- outputs/github-review.md
- PROGRESS.md
Do not:
- Push commits
- Merge pull requests
- Close issues
- Post comments
- Send Slack messages
- Modify source files
If an external action seems useful, draft it in outputs/recommended-actions.md and mark it as needing human review.
````

### Original block 87

````text
mkdir minimal-claude-loop
cd minimal-claude-loop

touch TASK.md
touch PROGRESS.md
touch LOOP_INSTRUCTIONS.md
mkdir outputs
touch outputs/daily-review.md
````

### Original block 88

````text
minimal-claude-loop/
├── TASK.md
├── PROGRESS.md
├── LOOP_INSTRUCTIONS.md
└── outputs/
    └── daily-review.md
````

### Original block 89

````text
# Daily Project Review Loop

## Goal
Review this project folder, summarize what changed, identify blockers, and produce a short daily review report.
## Expected Output
Each run should produce or update:
- `outputs/daily-review.md`
- `PROGRESS.md`
## Scope
Claude may inspect files in this workspace and write reports to the `outputs/` folder.
Claude should not modify source files, delete files, rename files, move files, send messages, open tickets, or interact with external systems during this first version of the loop.
````

### Original block 90

````text
# Loop Progress

## Current State
- Status: Manual setup
- Main objective: Produce a daily project review report
- Current focus: Validate the loop manually before scheduling
- Last updated:
## Last Run
- Date:
- Trigger:
- Summary:
- Files reviewed:
- Output produced:
## Open Items
-
## Blockers
-
## Needs Human Review
-
## Next Run Should
- Read `TASK.md`, `PROGRESS.md`, and `LOOP_INSTRUCTIONS.md`.
- Inspect the workspace.
- Write `outputs/daily-review.md`.
- Update this file before stopping.
## Decisions Made
- The loop starts with local file access only.
- The loop is allowed to write only to `outputs/daily-review.md` and `PROGRESS.md`.
- Scheduling should happen only after stable manual runs.
## Do Not Repeat
- Do not modify source files.
- Do not delete, rename, or move files.
- Do not create extra files unless explicitly instructed.
````

### Original block 91

````text
# Loop Instructions

You are running a daily project review loop.
## Before You Start
1. Read `TASK.md`.
2. Read `PROGRESS.md`.
3. Inspect the project folder.
4. Identify what changed, what is incomplete, and what needs human review.

## What You Should Do
Write a short daily review report to:
`outputs/daily-review.md`
The report should include:
- Summary of current project state
- Files or areas reviewed
- Meaningful changes found
- Blockers or unresolved questions
- Recommended next actions
After writing the report, update `PROGRESS.md` with:
- Date of this run
- Summary of what happened
- Files checked
- Output produced
- What the next run should do
- Anything that needs human review

## Safety Rules
- Do not delete files.
- Do not rename files.
- Do not move files.
- Do not modify source files.
- Do not send messages.
- Do not open tickets.
- Do not interact with external systems.
- Only write to `outputs/daily-review.md` and `PROGRESS.md`.
- If you are unsure whether an action is allowed, stop and ask for human review.

## Scheduled Run Policy
When this loop runs on a schedule:
- If there are meaningful changes, write a concise report to `outputs/daily-review.md`.
- If there are no meaningful changes, write a short "No meaningful changes" note.
- If human review is needed, mark it clearly in `PROGRESS.md`.
- If the same blocker appears in two consecutive runs, do not keep retrying. Escalate to human review.
- If verification fails twice, stop and mark the run as not accepted.
- Keep scheduled outputs short unless there is something important to report.
- Do not create extra files unless explicitly allowed.
- Do not modify any files except `outputs/daily-review.md` and `PROGRESS.md`.

## Verification Checklist
Before ending the run, verify the following:
### Required Files
- `outputs/daily-review.md` exists.
- `PROGRESS.md` exists.
### Required Report Sections
`outputs/daily-review.md` must include:
- Summary
- Files reviewed
- Meaningful changes
- Blockers or unresolved questions
- Recommended next actions
### State Update
`PROGRESS.md` must include:
- Date of the current run
- Summary of what happened
- Files checked
- Output produced
- What the next run should do
- Whether human review is needed
### Safety Boundary
Only these files may be modified:
- `outputs/daily-review.md`
- `PROGRESS.md`
If any other file was modified, stop and report the issue.

## Failure Policy
If verification fails:
1. If the failure is a missing section in `outputs/daily-review.md`, fix the report once.
2. If `PROGRESS.md` was not updated, update it once.
3. If any forbidden file was modified, stop immediately and report the issue.
4. If the same verification check fails twice, stop and mark the run as needing human review.
````

### Original block 92

````text
# Daily Review

No loop run has been completed yet.
````

### Original block 93

````text
Run the daily project review loop for this workspace.

Follow `LOOP_INSTRUCTIONS.md` exactly.
Before acting, read:
- `TASK.md`
- `PROGRESS.md`
- `LOOP_INSTRUCTIONS.md`
Then inspect the workspace, write the daily review to `outputs/daily-review.md`, update `PROGRESS.md`, run the verification checklist, and report what changed.
Do not modify any files except `outputs/daily-review.md` and `PROGRESS.md`.
````

### Original block 94

````text
outputs/daily-review.md
PROGRESS.md
````

### Original block 95

````text
Run a verification pass for the latest loop execution.

Check the output against the Verification Checklist in `LOOP_INSTRUCTIONS.md`.
Report:
1. Which checks passed
2. Which checks failed
3. Which files were modified
4. Whether the run is safe to accept
5. Whether human review is required
Do not modify any files during this verification pass
````

### Original block 96

````text
Verification result: ACCEPTED

Passed:
- outputs/daily-review.md exists
- Required report sections are present
- PROGRESS.md was updated
- No forbidden files were modified
Human review required:
- No
````

### Original block 97

````text
Verification result: NOT ACCEPTED

Failed:
- Missing "Blockers or unresolved questions" section in outputs/daily-review.md
Human review required:
- No, this is safe to fix once
````

### Original block 98

````text
Verification result: NOT ACCEPTED

Failed:
- Forbidden file modified: src/config.ts
Human review required:
- Yes
````

### Original block 99

````text
Test run 1:
Initial workspace only.

Test run 2:
Add a notes.md file and check whether the loop notices it.

Test run 3:
Add a blocker to PROGRESS.md and check whether the loop carries it forward.

Test run 4:
Add a file inside outputs/ and check whether the loop avoids treating generated outputs as source material.

Test run 5:
Add an unresolved question and check whether it appears under human review.
````

### Original block 100

````text
- Did Claude read PROGRESS.md?
- Did Claude update PROGRESS.md?
- Did the report stay concise?
- Did it avoid modifying forbidden files?
- Did it correctly mark human review when needed?
- Did it avoid repeating stale information?
````

### Original block 101

````text
/loop 24h Run the daily project review loop for this workspace.

Follow `LOOP_INSTRUCTIONS.md` exactly.
Read `TASK.md`, `PROGRESS.md`, and `LOOP_INSTRUCTIONS.md` before acting.
Write the daily review to `outputs/daily-review.md`.
Update `PROGRESS.md` before stopping.
Run the verification checklist.
If there are no meaningful changes, keep the report short.
If human review is needed, clearly mark it in `PROGRESS.md`.
Do not modify any files except `outputs/daily-review.md` and `PROGRESS.md`.
````

### Original block 102

````text
/loop 15m Run the daily project review loop for this workspace. Follow `LOOP_INSTRUCTIONS.md` exactly.
````

### Original block 103

````text
Useful output:
Did the loop produce something worth reading?

Noise:
Did the loop create unnecessary summaries or false alarms?

Missed issue:
Did the loop fail to notice something important?

Boundary respect:
Did the loop only modify allowed files?

State quality:
Did PROGRESS.md remain short and useful?
````

### Original block 104

````text
## Loop Review Notes

- 2026-06-24: Output useful. Report was concise.
- 2026-06-25: Report repeated previous summary. Added instruction to focus on changes since last run.
- 2026-06-26: Missed new file in docs/. Added docs/ to inspection scope.
````

### Original block 105

````text
Git:
Inspect recent commits and include them in the daily review.

GitHub:
Read open issues, pull requests, and CI status.

Slack:
Draft a daily update, but do not send it automatically.

Linear or Jira:
Read assigned tickets and draft suggested status updates.

Worktree:
Allow code edits only inside an isolated branch or worktree.
````

### Original block 106

````text
1. Local file loop
2. Git read-only context
3. GitHub read-only context
4. Draft external updates
5. Human-approved actions
6. Automated low-risk actions only after repeated success
````

### Original block 107

````text
## GitHub Policy

Allowed:
- Read open issues
- Read pull request status
- Read CI status
- Summarize findings in `outputs/github-review.md`
Not allowed:
- Push commits
- Merge pull requests
- Close issues
- Add comments
- Change labels
- Modify milestones
````

### Original block 108

````text
## Slack Policy

Allowed:
- Draft a message in `outputs/slack-draft.md`
Not allowed:
- Send messages
- Mention users
- Post to channels
- Create alerts
````

### Original block 109

````text
- Scheduling before manual testing
- No state file
- State file too long
- No verification checklist
- Vague stop condition
- Broad write permissions
- Too many tools too early
- No human review window
- No audit trail
- Treating the first successful run as proof of reliability
````

### Original block 110

````text
- A clear trigger
- A stable task file
- A persistent state file
- A constrained instruction file
- A predictable output location
- A verification checklist
- A human review path
- A permission boundary
````

### Original block 111

````text
Trigger:
Manual prompt first, `/loop` later.

Context:
TASK.md, PROGRESS.md, LOOP_INSTRUCTIONS.md.

Action:
Claude inspects the workspace and writes a daily report.

Verification:
Claude checks required files, required sections, state update, and safety boundaries.

State:
PROGRESS.md records what happened and what should happen next.

Review:
A human inspects outputs before the loop gains more autonomy.
````

### Original quotation/output 1

````text
A Claude loop is not just “ask Claude again.” It is a repeatable workflow around Claude: trigger, context, action, verification, state update, and decision. The model performs the work, but the loop controls how the work continues.
````

### Original quotation/output 2

````text
The simplest reliable loop should be deliberately small. It should read context, do one constrained job, write the result somewhere predictable, update state, and stop. More autonomy can be added later, but the first objective is repeatability.
````

### Original quotation/output 3

````text
Loop engineering is the practice of designing those boundaries: what starts the loop, what Claude can access, what Claude can change, how progress is recorded, how results are verified, and when a human must review the output.
````

### Original quotation/output 4

````text
A Claude loop is most useful when the task repeats, the output can be checked, and the next run benefits from what happened in the previous run. Without those three conditions, a loop often becomes a more expensive version of a normal prompt.
````

### Original quotation/output 5

````text
The first loop should usually be read-heavy and write-light. Let Claude inspect, summarize, classify, and draft before allowing it to modify important files or external systems.
````

### Original quotation/output 6

````text
A loop is not successful because it runs automatically. It is successful when it produces useful work with less supervision than manual prompting, while staying within clear cost, permission, and review boundaries.
````

### Original quotation/output 7

````text
The first Claude loop should be intentionally simple. It should read context, perform one constrained task, write output to a predictable location, update state, and stop. The objective is not full autonomy. The objective is repeatability.
````

### Original quotation/output 8

````text
PROGRESS.md is the loop’s memory interface. Claude should read it before acting and update it before stopping. If the state file is missing, stale, or too long, the loop loses continuity.
````

### Original quotation/output 9

````text
Verification should exist from the first loop, even if it starts as a checklist. A loop that cannot check its own boundaries will eventually create review work instead of reducing it.
````

### Original quotation/output 10

````text
PROGRESS.md is not just a note file. It is the memory interface of the loop. Each run should read it before acting and update it before stopping.
````

### Original quotation/output 11

````text
A useful state file is closer to a control panel than an archive. It should tell the next run where the loop is, what matters now, what is blocked, and what should happen next.
````

### Original quotation/output 12

````text
If Claude needs it to decide the next action, keep it in PROGRESS.md.
If a human may want to inspect it later, store it in outputs/.
````

### Original quotation/output 13

````text
Persistent state is also a control surface. By editing PROGRESS.md, you can redirect the next run, block unsafe actions, record decisions, and keep Claude from repeating known mistakes.
````

### Original quotation/output 14

````text
A Claude loop should not stop because Claude says it is finished. It should stop because a concrete condition has been checked: a file exists, a required structure is present, a test passes, a checklist is complete, or a human review is required.
````

### Original quotation/output 15

````text
The checker should not be another vague reviewer. It should evaluate the output against specific criteria. A second opinion is useful only when it has a standard to check against.
````

### Original quotation/output 16

````text
A verifier needs a pass/fail standard. “Review this” is too weak. “Check these seven conditions and mark the run accepted only if all seven pass” is a much stronger loop boundary.
````

### Original quotation/output 17

````text
The schedule should follow the task, not the other way around. A loop should run only as often as new information is likely to appear or action is likely to be useful.
````

### Original quotation/output 18

````text
Scheduling increases the value of a loop, but it also increases the cost of mistakes. The safest scheduled loops begin with read-only analysis, draft outputs, explicit verification, and human review.
````

### Original quotation/output 19

````text
Trigger → Context → Action → Verification → State update → Stop / escalate / repeat
````

### Original quotation/output 20

````text
Tools make loops useful beyond the local folder, but every new tool expands the loop’s blast radius. A good loop should gain permissions slowly: read first, draft second, write only after repeated successful runs.
````

### Original quotation/output 21

````text
A connector should start as an observation channel, not an action channel. Let the loop read external systems and draft recommendations before allowing it to update or execute anything.
````

### Original quotation/output 22

````text
Do not rely only on prompt instructions for safety. Use tool permissions, scoped access, and human approval gates so the loop cannot perform high-impact actions just because the prompt drifted or the task changed.
````

### Original quotation/output 23

````text
The first complete loop should be boring by design. It should do one useful thing: write to known files, verify its own boundaries, and leave a clear state for the next run.
````

### Original quotation/output 24

````text
A good state file is both a memory and a control surface. It tells Claude what happened, what matters now, what is blocked, and what should not be repeated.
````

### Original quotation/output 25

````text
Manual verification is the bridge between a demo and a working loop. The loop should not be scheduled until its output, state update, and safety boundaries are verified across multiple runs.
````